In [1]:
import numpy as np
import torch
import torch.nn as nn

from models.model import End2EndModel
from experiments.purity.purity_model import PurityModel

torch.set_printoptions(sci_mode=False)

from experiments.utils import get_device, get_dataset

/home/jack/anaconda3/envs/playing_cards/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
device = get_device(cpu=True)

N_CONCEPT, N_CLASSES, val_loader, concept_index_to_string = get_dataset("three", return_points=False, return_orig=False, return_path=False, shuffle=True)

XtoCtoY = End2EndModel(num_classes=6, num_concepts=N_CONCEPT, use_sigmoid=False, use_modulelist=False)
XtoCtoY.load_state_dict(torch.load("./../saves/converted/independent-three-seed-950.pth"))
XtoC = XtoCtoY.x_to_c_model
XtoC.to(device)
XtoC.eval()

purity_model = PurityModel(num_concepts=XtoC.out_fc.out_features)
purity_model.load_state_dict(torch.load("./../saves/independent-three-seed-950_OIS_model.pth"))
purity_model.to(device)
purity_model.eval()

print("models loaded")

Device: cpu NVIDIA GeForce RTX 3050 Ti Laptop GPU
models loaded


# True concept values

In [20]:
imageId = 1

images, labels, concepts = next(iter(val_loader))
image = images[imageId].to(device)

print("True concepts:", concepts[imageId])

# print concept as string if prediction is >= 0.5
for idx, i in enumerate(concepts[imageId]):
    if i.item() >= 0.5:
        print(f"id: {idx},", f"concept name: {concept_index_to_string(idx)},")

True concepts: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
id: 9, concept name: 4D,
id: 22, concept name: 7H,
id: 32, concept name: 10C,


# X to C model concept predictions

In [21]:
image = image.unsqueeze(0)

prediction = XtoC(image)
pred_concepts_readable = torch.nn.Sigmoid()(prediction)

print(pred_concepts_readable)

# print concept as string if prediction is >= 0.5
for idx, i in enumerate(pred_concepts_readable[0]):
    if i.item() >= 0.5:
        print(f"id: {idx},", f"concept name: {concept_index_to_string(idx)},", f"sig value: {i.item()}")

tensor([[    0.0000,     0.0000,     0.0001,     0.0000,     0.0000,     0.0000,
             0.0000,     0.0000,     0.0000,     1.0000,     0.0007,     0.0000,
             0.0000,     0.0000,     0.0001,     0.0000,     0.0000,     0.0004,
             0.0000,     0.0000,     0.0000,     0.0000,     1.0000,     0.0000,
             0.0000,     0.0000,     0.0001,     0.0000,     0.0000,     0.0001,
             0.0000,     0.0000,     1.0000,     0.0000,     0.0000,     0.0000,
             0.0000,     0.0000,     0.0000,     0.0000,     0.0000,     0.0000,
             0.0000,     0.0000,     0.0000,     0.0000,     0.0000,     0.0000,
             0.0000,     0.0000,     0.0000,     0.0000]],
       grad_fn=<SigmoidBackward0>)
id: 9, concept name: 4D, sig value: 0.9999592304229736
id: 22, concept name: 7H, sig value: 0.9999839067459106
id: 32, concept name: 10C, sig value: 1.0


# MLP concept predictions from single concepts

In [34]:
one_hot = nn.functional.one_hot(torch.arange(0, concepts.size(1)), num_classes=concepts.size(1)).to(device)
concepts_in = []
for i in range(prediction.size(0)):
    concepts_in.append(torch.mul(prediction[i], one_hot))

# x_to_c_model concept predictions are turned into one hot like vectors
# each concept prediction is in an array of size (0, num_concepts) where all but that concept has the value of 0
# the new batch size for purity_model training is original batch size * num_concepts
concepts_in = torch.cat(concepts_in)

# repeate concepts to use as lables for concepts_in
target_concepts = torch.repeat_interleave(prediction.detach(), concepts_in.size(1), dim=0)

outputs = purity_model(concepts_in)
outputs_readable = torch.nn.Sigmoid()(outputs)

for idx, i in enumerate(outputs_readable):
    print(f"input concept: {concept_index_to_string(idx)}")
    print(i)

    # print concept as string if prediction is >= 0.5
    for idy, j in enumerate(i):
        if j.item() >= 0.5:
            print(f"id: {idy},", f"concept name: {concept_index_to_string(idy)},", f"sig value: {j.item()}")

input concept: 2C
tensor([0.5852, 0.5756, 0.4845, 0.5321, 0.5540, 0.6151, 0.5096, 0.5805, 0.4312,
        0.5193, 0.6930, 0.3868, 0.5331, 0.2785, 0.2960, 0.6715, 0.3836, 0.4899,
        0.5121, 0.5582, 0.4632, 0.2986, 0.5774, 0.3742, 0.3567, 0.4151, 0.6148,
        0.3722, 0.3556, 0.3277, 0.5110, 0.5439, 0.6193, 0.3541, 0.5357, 0.5878,
        0.6810, 0.3098, 0.3964, 0.5769, 0.6533, 0.6150, 0.4164, 0.4389, 0.5924,
        0.5856, 0.3742, 0.5111, 0.3878, 0.4671, 0.3788, 0.5114],
       grad_fn=<UnbindBackward0>)
id: 0, concept name: 2C, sig value: 0.5852299332618713
id: 1, concept name: 2D, sig value: 0.5755770206451416
id: 3, concept name: 2S, sig value: 0.5321239233016968
id: 4, concept name: 3C, sig value: 0.5540207028388977
id: 5, concept name: 3D, sig value: 0.6151344180107117
id: 6, concept name: 3H, sig value: 0.5096113085746765
id: 7, concept name: 3S, sig value: 0.5804990530014038
id: 9, concept name: 4D, sig value: 0.519286572933197
id: 10, concept name: 4H, sig value: 0.69304